In [ ]:
# ============================================================
# ACTIVITY 3: HYBRID MODEL USING OPTUNA-TUNED SVM + RANDOM FOREST
# Continue after Activity 2
# Uses best_params from Optuna study
# ============================================================
from sklearn.ensemble import RandomForestClassifier, VotingClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42
)

In [ ]:
# ============================================================
# 3. Hybrid Voting Model
# SVM + Random Forest
# ============================================================
# ============================================================
# Build SVM using Optuna best parameters
# ============================================================

optuna_svm = SVC(
    kernel=best_params["kernel"],
    C=best_params["C"],
    gamma=best_params.get("gamma", "scale"),
    class_weight=best_params["class_weight"],
    probability=True,      # needed for soft voting
    random_state=42
)

hybrid_model = VotingClassifier(
    estimators=[
        ("svm", optuna_svm),
        ("rf", rf_model)
    ],
    voting="soft"   # use probability averaging
)

In [ ]:
# ============================================================
# 4. Full Pipeline
# ============================================================

hybrid_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("hybrid_model", hybrid_model)
])

In [ ]:
# ============================================================
# 5. Train Hybrid Model
# ============================================================

hybrid_pipeline.fit(X_train, y_train)

hybrid_pred = hybrid_pipeline.predict(X_test)

In [ ]:
# ============================================================
# 6. Evaluate Hybrid Model
# ============================================================

hybrid_accuracy = accuracy_score(y_test, hybrid_pred)
hybrid_precision = precision_score(y_test, hybrid_pred)
hybrid_recall = recall_score(y_test, hybrid_pred)
hybrid_f1 = f1_score(y_test, hybrid_pred)

print("===== HYBRID MODEL: OPTUNA SVM + RANDOM FOREST =====")
print("Accuracy :", round(hybrid_accuracy,4))
print("Precision:", round(hybrid_precision,4))
print("Recall   :", round(hybrid_recall,4))
print("F1 Score :", round(hybrid_f1,4))


In [ ]:
# ============================================================
# 7. Final Comparison
# baseline metrics from Activity 1:
# accuracy, precision, recall, f1
#
# optuna metrics from Activity 2:
# enhanced_accuracy, enhanced_precision,
# enhanced_recall, enhanced_f1
# ============================================================

comparison = pd.DataFrame({
    "Model":[
        "Baseline SVM",
        "Optuna Enhanced SVM",
        "Hybrid Optuna SVM + RF"
    ],
    "Accuracy":[
        accuracy,
        enhanced_accuracy,
        hybrid_accuracy
    ],
    "Precision":[
        precision,
        enhanced_precision,
        hybrid_precision
    ],
    "Recall":[
        recall,
        enhanced_recall,
        hybrid_recall
    ],
    "F1 Score":[
        f1,
        enhanced_f1,
        hybrid_f1
    ]
})

comparison

In [ ]:
# ============================================================
# 8. Plot Final Results
# ============================================================

import matplotlib.pyplot as plt

comparison_plot = comparison.set_index("Model")

comparison_plot.plot(kind="bar", figsize=(12,6))

plt.title("Baseline vs Optuna vs Hybrid Model")
plt.ylabel("Score")
plt.ylim(0,1)
plt.xticks(rotation=15)
plt.grid(axis="y")
plt.show()